# Retail Sales Performance & Data Analysis

## Python Data Analysis Project — 2004 to 2006

#### Project Objectives & Workflow:
#### 1. Data Acquisition: Load and structurally inspect the transactional sales dataset using Pandas.
#### 2. Data Wrangling: Perform data cleaning, handling missing anomalies, and preprocessing.
#### 3. Data Transformation: Standardize and prepare the data pipeline for analytical modeling.
#### 4. Exploratory Data Analysis (EDA): Identify underlying trends, sales behaviors, and seasonal patterns.
#### 5. Business Metric Analysis: Conduct statistical deep-dives into key corporate variables (Sales & Profit).
#### 6. Data Visualization: Develop clear, insightful charts to communicate data-driven recommendations.

#### Libraries Used: pandas | numpy | matplotlib | plotly 




---
## Step 1: Load the Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_excel('OmairSalesDataProject.xlsx')

print('Dataset loaded successfully!')

print(df.head())


print(df.info())





---
## Step 2: Data Cleaning & Preprocessing





In [ ]:
df['Date']=pd.to_datetime(df["Date"],format='%Y-%m-%d', errors='coerce')
df['Cost']=pd.to_numeric(df['Cost'],errors='coerce')
print(df.isnull().sum())
print('-------------------------')
print(df.duplicated().sum())
print('-------------------------')
print(df['Cost'].nlargest())
print('-------------------------')
for col in ['City','Rep','Store','Prod']:
    print(df[col].unique())


In [ ]:
#data preprocessing

df.loc[df['Cost']> 1000000,'Cost']= np.nan
df['Cost']=df['Cost'].fillna(df['Cost'].mean())
df['Rep']=df['Rep'].replace('Mjeeed', 'Mjeed')
df['Store']=df['Store'].replace('Lulu Hyper','Lulu')


---
## Step 3: Data Transformation

In [ ]:
df['Profit'] = df['Sale'] - df['Cost']
df['Profit_Margin %'] =( df['Profit'] / df['Sale'])

df=df.drop(columns=["Profit_Margin"],errors="ignore")
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['Year'] =df['Year'].replace({2015:2005,2014:2004})
display(df.head())


---
## Step 4: Exploratory Data Analysis (EDA)

In [ ]:
# Dataset overview
print(f'Total Records: {len(df):,}')
print(f'Cities       : {sorted(df["City"].unique())}')
print(f'Sales Reps   : {sorted(df["Rep"].unique())}')
print(f'Stores       : {sorted(df["Store"].unique())}')
print(f'Products     : {sorted(df["Prod"].unique())}')
print(f'Years        : {sorted(df["Year"].unique())}')

In [ ]:
# Distribution by category
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Distribution by Category', fontsize=16, fontweight='bold', y=1.01)

colors = ['#4361ee','#3a0ca3','#7209b7','#f72585','#4cc9f0']

cats = [('City','City'), ('Rep','Sales Rep'), ('Store','Store'), ('Prod','Product')]
for ax, (col, label) in zip(axes.flat, cats):
    counts = df[col].value_counts()
    ax.bar(counts.index, counts.values, color=colors[:len(counts)])
    ax.set_title(f'Transactions by {label}', fontweight='bold')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=30)
    for bar, v in zip(ax.patches, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, f'{v:,}',
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Yearly summary
yearly = df.groupby('Year').agg(
    Transactions=('Sale','count'),
    Total_Sales=('Sale','sum'),
    Total_Cost=('Cost','sum'),
    Total_Profit=('Profit','sum'),
    Avg_Margin=('Profit_Margin %','mean')
).round(2)

print('=== Yearly Summary ===')
print(yearly.to_string())

### 🧹 Data Wrangling & Transformation (Success)
###  Pipeline Update: The dataset has been successfully cleaned, transformed, and validated. High data integrity is confirmed, and the pipeline is now fully prepared for advanced statistical analysis and business metrics interpretation.

---
## Step 5: Statistical Analysis (Sales & Profit)


In [ ]:
# Descriptive statistics
print('=== Descriptive Statistics ===')
stats = df[['Cost','Sale','Profit','Profit_Margin %']].describe().round(2)
print(stats)

print('\n=== Key KPIs ===')
print(f'Total Revenue  : SAR {df["Sale"].sum():>15,.2f}')
print(f'Total Cost     : SAR {df["Cost"].sum():>15,.2f}')
print(f'Total Profit   : SAR {df["Profit"].sum():>15,.2f}')
print(f'Avg Margin     :     {df["Profit_Margin %"].mean()*100:>14.2f}%')
print(f'Profitable Txns:     {(df["Profit"]>0).sum():>14,} / {len(df):,}')

In [ ]:
# Sales & Profit by Product
by_prod = df.groupby('Prod').agg(
    Total_Sales=('Sale','sum'),
    Total_Profit=('Profit','sum'),
    Avg_Margin=('Profit_Margin %','mean'),
    Transactions=('Sale','count')
).round(2).sort_values('Total_Sales', ascending=False)

print('=== Sales & Profit by Product ===')
print(by_prod.to_string())

In [ ]:
# Sales & Profit by City
by_city = df.groupby('City').agg(
    Total_Sales=('Sale','sum'),
    Total_Profit=('Profit','sum'),
    Avg_Margin=('Profit_Margin %','mean')
).round(2).sort_values('Total_Profit', ascending=False)

print('=== Sales & Profit by City ===')
print(by_city.to_string())

In [ ]:
# Sales Reps performance
by_rep = df.groupby('Rep').agg(
    Total_Sales=('Sale','sum'),
    Total_Profit=('Profit','sum'),
    Transactions=('Sale','count'),
    Avg_Sale_per_Txn=('Sale','mean')
).round(2).sort_values('Total_Profit', ascending=False)

print('=== Sales Rep Performance ===')
print(by_rep.to_string())


## Step 6: Visualizations

In [ ]:
# Sales vs Profit by Product
from plotly.subplots import make_subplots 
import plotly.graph_objects as go
fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Total Sales by Product', 'Total Profit by Product'])

by_prod_sorted = by_prod.sort_values('Total_Sales', ascending=True)

fig.add_trace(go.Bar(
    x=by_prod_sorted['Total_Sales'], y=by_prod_sorted.index,
    orientation='h', marker_color=['#4361ee','#7209b7','#f72585','#4cc9f0'],
    name='Sales'
), row=1, col=1)

by_prod_profit = by_prod.sort_values('Total_Profit', ascending=True)
colors_profit = ['#ef233c' if v < 0 else '#2dc653' for v in by_prod_profit['Total_Profit']]
fig.add_trace(go.Bar(
    x=by_prod_profit['Total_Profit'], y=by_prod_profit.index,
    orientation='h', marker_color=colors_profit,
    name='Profit'
), row=1, col=2)

fig.update_layout(title='Sales & Profit by Product', height=350, showlegend=False,
    plot_bgcolor='white', paper_bgcolor='white')
fig.show()

In [ ]:
# Profit by City
by_city_sorted = by_city.sort_values('Total_Profit')
clrs = ['#ef233c' if v < 0 else '#4361ee' for v in by_city_sorted['Total_Profit']]

fig = go.Figure(go.Bar(
    x=by_city_sorted['Total_Profit'],
    y=by_city_sorted.index,
    orientation='h',
    marker_color=clrs,
    text=[f'SAR {v:,.0f}' for v in by_city_sorted['Total_Profit']],
    textposition='outside'
))
fig.update_layout(
    title='Total Profit by City',
    xaxis_title='Profit (SAR)',
    height=380,
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig.show()

In [ ]:
# Sales Rep Performance
by_rep_sorted = by_rep.sort_values('Total_Sales')

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Total Sales by Rep', 'Total Profit by Rep'])

fig.add_trace(go.Bar(
    x=by_rep_sorted['Total_Sales'], y=by_rep_sorted.index,
    orientation='h',
    marker_color='#4361ee',
    name='Sales'
), row=1, col=1)

by_rep_profit = by_rep.sort_values('Total_Profit')
clrs2 = ['#ef233c' if v < 0 else '#2dc653' for v in by_rep_profit['Total_Profit']]
fig.add_trace(go.Bar(
    x=by_rep_profit['Total_Profit'], y=by_rep_profit.index,
    orientation='h',
    marker_color=clrs2,
    name='Profit'
), row=1, col=2)

fig.update_layout(title='Sales Rep Performance', height=350, showlegend=False,
    plot_bgcolor='white', paper_bgcolor='white')
fig.show()

In [ ]:
# Monthly Sales Trend
monthly = df.groupby(['Year','Month']).agg(
    Sales=('Sale','sum'),
    Profit=('Profit','sum')
).reset_index().sort_values(['Year','Month'])
monthly['YearMonth']=monthly['Year'].astype(str)+"-"+monthly['Month'].astype(str)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=monthly['YearMonth'], y=monthly['Sales'],
    mode='lines+markers', name='Sales',
    line=dict(color='#4361ee', width=2)
))
fig.add_trace(go.Scatter(
    x=monthly['YearMonth'], y=monthly['Profit'],
    mode='lines+markers', name='Profit',
    line=dict(color='#2dc653', width=2)
))
fig.update_layout(
    title='Monthly Sales & Profit Trend (2004–2006)',
    xaxis_title='Month',
    yaxis_title='Amount (SAR)',
    height=420,
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)
fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
# Margin Distribution
import plotly.express as px
fig = px.histogram(
    df, x='Profit_Margin %', nbins=40,
    title='Distribution of Profit Margin',
    color_discrete_sequence=['#7209b7'],
    labels={'Profit_Margin %': 'Profit Margin'}
)
fig.add_vline(x=df['Profit_Margin %'].mean(), line_dash='dash',
              line_color='red', annotation_text=f" Avg: {df['Profit_Margin %'].mean():.2%}")
fig.update_layout(height=380, plot_bgcolor='white', paper_bgcolor='white')
fig.show()

In [ ]:
# Profit Heatmap: Store vs Product
pivot = df.pivot_table(values='Profit', index='Store', columns='Prod', aggfunc='sum').round(0)

fig = px.imshow(
    pivot,
    text_auto=True,
    color_continuous_scale='RdYlGn',
    title='Profit Heatmap: Store × Product (SAR)',
    labels=dict(color='Profit (SAR)')
)
fig.update_layout(height=380)
fig.show()

In [ ]:
# Profit Category Distribution
bins = [-float('inf'), 0, 500, 2000, float('inf')]
labels = ['Loss', 'Low Profit', 'Medium Profit', 'High Profit']
df['Profit_Category'] = pd.cut(df['Profit'], bins=bins, labels=labels)

profit_cat = df['Profit_Category'].value_counts().reset_index()
profit_cat.columns = ['Category', 'Count']

color_map = {
    'Loss':'#ef233c',
    'Low Profit':'#fca311',
    'Medium Profit':'#4361ee',
    'High Profit':'#2dc653'
}

fig = px.pie(
    profit_cat, names='Category', values='Count',
    title='Transactions by Profit Category',
    color='Category',
    color_discrete_map=color_map,
    hole=0.4
)
fig.update_traces(textposition='outside', textinfo='percent+label')
fig.update_layout(height=400)
fig.show()

In [ ]:
# Scatter: Sale vs Profit colored by Product
fig = px.scatter(
    df.sample(600, random_state=42),
    x='Sale', y='Profit',
    color='Prod',
    title='Sale vs Profit by Product (sample 600)',
    labels={'Sale':'Sale Amount (SAR)', 'Profit':'Profit (SAR)'},
    opacity=0.6
)
fig.add_hline(y=0, line_dash='dash', line_color='red')
fig.update_layout(height=420, plot_bgcolor='white', paper_bgcolor='white')
fig.show()

## 💡 Key Finding (Product & Regional Insights)
### The visualization reveals a highly skewed distribution in sales, with specific regions and product categories capturing the majority of market volume. 
### This indicates high market concentration and highlights our primary revenue drivers.

---
##  Summary of Key Insights

In [ ]:
top_city   = by_city['Total_Profit'].idxmax()
top_rep    = by_rep['Total_Profit'].idxmax()
top_prod   = by_prod['Total_Profit'].idxmax()
top_store  = df.groupby('Store')['Profit'].sum().idxmax()
loss_pct   = (df['Profit'] < 0).mean() * 100


print('       KEY INSIGHTS — OMAIR SALES DATA')
print('=' * 50)
print(f'  Total Revenue  : SAR {df["Sale"].sum():>12,.0f}')
print(f'  Total Profit   : SAR {df["Profit"].sum():>12,.0f}')
print(f'  Avg Margin     : {df["Profit_Margin %"].mean()*100:>14.1f}%')
print(f'  Loss Txns      : {loss_pct:>14.1f}%  of all transactions')
print('-' * 50)
print(f'  Top City    : {top_city}')
print(f'  Top Rep     : {top_rep}')
print(f'  Top Product : {top_prod}')
print(f'   Top Store   : {top_store}')
print('=' * 50)